In [4]:
import numpy as np
import matplotlib as plt
from pathlib import Path
import sys

In [1]:
# Paths + autoreload so edits in src/ are picked up without restarting the kernel


%load_ext autoreload
%autoreload 2

def find_project_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / "src").exists() and (p / "data").exists():
            return p
    raise RuntimeError("Project root not found. Open the notebook inside the project tree.")

ROOT = find_project_root()
sys.path.append(str(ROOT))          # so `from src...` works
DATA_DIR = ROOT / "data"

ROOT, DATA_DIR


(PosixPath('/Users/chedliabdennebi/Desktop/ADA/projet'),
 PosixPath('/Users/chedliabdennebi/Desktop/ADA/projet/data'))

In [5]:
from src.utils.loading_data import (
    load_stocks, load_etfs, read_metadata,
    print_basic_info, describe_symbols, describe_missing
)

# Paths
stocks_path = DATA_DIR / "stocks"
etfs_path   = DATA_DIR / "etfs"
meta_path   = DATA_DIR / "symbols_valid_meta.csv"

# Load
stocks_df = load_stocks(stocks_path)
etfs_df   = load_etfs(etfs_path)
meta_df   = read_metadata(meta_path)

# Quick confirmation
(stocks_df.shape, etfs_df.shape, meta_df.shape)



((24197442, 8), (3954316, 8), (8049, 12))

In [7]:
print_basic_info("stocks", stocks_df)
print_basic_info("etfs", etfs_df)
print_basic_info("meta", meta_df)



=== STOCKS ===
shape: (24197442, 8)
columns: ['Symbol', 'Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
date range: 1962-01-02 00:00:00  ->  2020-04-02 00:00:00
symbols: 5884

=== ETFS ===
shape: (3954316, 8)
columns: ['Symbol', 'Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
date range: 1986-04-03 00:00:00  ->  2020-04-01 00:00:00
symbols: 2165

=== META ===
shape: (8049, 12)
columns: ['Nasdaq Traded', 'Symbol', 'Security Name', 'Listing Exchange', 'Market Category', 'ETF', 'Round Lot Size', 'Test Issue', 'Financial Status', 'CQS Symbol', 'NASDAQ Symbol', 'NextShares']
symbols: 8049


In [8]:
stocks_df.head()

,Symbol,Date,Open,High,Low,Close,Adj Close,Volume
0,A,1999-11-18,32.546494,35.765381,28.612303,31.473534,27.068665,62546300.0
1,A,1999-11-19,30.713520,30.758226,28.478184,28.880543,24.838577,15234100.0
2,A,1999-11-22,29.551144,31.473534,28.657009,31.473534,27.068665,6577800.0
3,A,1999-11-23,30.400572,31.205294,28.612303,28.612303,24.607880,5975600.0
4,A,1999-11-24,28.701717,29.998211,28.612303,29.372318,25.261524,4843200.0


In [10]:
from src.utils.loading_data import create_symbol_dfs_from_folder

stocks_ns = create_symbol_dfs_from_folder(
    DATA_DIR / "stocks",
    prefix="stock_",
    scope=globals(),
    max_vars=None  # or 20 to limit
)


✅ Created 5884 DataFrames from /Users/chedliabdennebi/Desktop/ADA/projet/data/stocks


In [11]:
etfs_ns = create_symbol_dfs_from_folder(
    DATA_DIR / "etfs",
    prefix="etf_",
    scope=globals(),
    max_vars=None
)


✅ Created 2165 DataFrames from /Users/chedliabdennebi/Desktop/ADA/projet/data/etfs


In [23]:
from src.utils.preprocessing_data import nan_zero_report,check_duplicates,check_negative_prices

nan_zero_report(stocks_df)  





,column,missing_count,missing_%,zero_count,zero_%
0,Volume,649,0.0,1399027.0,5.78
1,Open,649,0.0,1179871.0,4.88
2,High,649,0.0,36.0,0.00
3,Low,649,0.0,36.0,0.00
4,Close,649,0.0,36.0,0.00
5,Adj Close,649,0.0,36.0,0.00
6,Symbol,0,0.0,NaN,NaN
7,Date,0,0.0,NaN,NaN


In [21]:
check_duplicates(stocks_df) 


np.int64(0)

In [25]:
check_negative_prices(stocks_df)

,Symbol,Date,Open,High,Low,Close,Adj Close,Volume
35636,AAN,1982-11-04,0.000000,1.024691,1.012346,1.012346,-1.263439e+21,9182700.0
35637,AAN,1982-11-05,0.000000,1.049383,1.024691,1.024691,-1.278846e+21,2370600.0
35638,AAN,1982-11-08,0.000000,1.061728,1.037037,1.037037,-1.294253e+21,1174500.0
35639,AAN,1982-11-09,0.000000,1.061728,1.037037,1.037037,-1.294253e+21,1344600.0
35640,AAN,1982-11-10,0.000000,1.098765,1.074074,1.074074,-1.340476e+21,819400.0
...,...,...,...,...,...,...,...,...
22345138,UN,1986-05-09,2.885417,2.904167,2.885417,2.891667,-4.561187e-04,774000.0
22345139,UN,1986-05-12,2.918750,2.933333,2.918750,2.933333,-4.626913e-04,768000.0
22345140,UN,1986-05-13,2.916667,2.916667,2.854167,2.875000,-4.534899e-04,768000.0
22345141,UN,1986-05-14,2.875000,2.904167,2.862500,2.904167,-4.580905e-04,1554000.0
